# COVID-19 Data Exploration on Filtered Data 

## Import required libraries.
- Check if kernel is in the correct environment

In [11]:
import pandas as pd
import numpy as np

import sys
print(sys.prefix)
print(sys.executable)

c:\Users\Dell\Documents\MSc AAI\PFAI - WM9QF\Individual Assessment\WM9QF-Task1-DataInsightsDashboard\venv
c:\Users\Dell\Documents\MSc AAI\PFAI - WM9QF\Individual Assessment\WM9QF-Task1-DataInsightsDashboard\venv\Scripts\python.exe


## Load the Dataset

In [12]:
FILE_PATH = "filtered_covid_data.csv"
print(FILE_PATH)

filtered_covid_data.csv


In [13]:
df = pd.read_csv(FILE_PATH)

## Data Exploration

In [14]:
df.head()

,gdp_per_capita,continent,new_cases,new_deaths,total_cases,total_deaths,iso_code,location,date,population
0,1803.99,Asia,0.0,0.0,0.0,0.0,AFG,Afghanistan,2020-01-05,41128772
1,1803.99,Asia,0.0,0.0,0.0,0.0,AFG,Afghanistan,2020-01-06,41128772
2,1803.99,Asia,0.0,0.0,0.0,0.0,AFG,Afghanistan,2020-01-07,41128772
3,1803.99,Asia,0.0,0.0,0.0,0.0,AFG,Afghanistan,2020-01-08,41128772
4,1803.99,Asia,0.0,0.0,0.0,0.0,AFG,Afghanistan,2020-01-09,41128772


Get number of rows and columns in the dataset

In [15]:
print(f"Dataset Shape", {df.shape})

Dataset Shape {(429435, 10)}


### Checking for Missing Values - Before preprocessing

Checking if NULL values are present in the dataset.

Calculating memory usage to check if Categorical will actually reduce memory usage

In [16]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429435 entries, 0 to 429434
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   gdp_per_capita  328292 non-null  float64
 1   continent       402910 non-null  object 
 2   new_cases       410159 non-null  float64
 3   new_deaths      410608 non-null  float64
 4   total_cases     411804 non-null  float64
 5   total_deaths    411804 non-null  float64
 6   iso_code        429435 non-null  object 
 7   location        429435 non-null  object 
 8   date            429435 non-null  object 
 9   population      429435 non-null  int64  
dtypes: float64(5), int64(1), object(4)
memory usage: 111.8 MB


Checking how many NULL values are present

In [17]:
df.isnull().sum()

gdp_per_capita    101143
continent          26525
new_cases          19276
new_deaths         18827
total_cases        17631
total_deaths       17631
iso_code               0
location               0
date                   0
population             0
dtype: int64

The type of values each column holds
- To check if there is need to change the data type for ease of application

In [18]:
print("Data Types of Each Column:")
df.dtypes 

Data Types of Each Column:


gdp_per_capita    float64
continent          object
new_cases         float64
new_deaths        float64
total_cases       float64
total_deaths      float64
iso_code           object
location           object
date               object
population          int64
dtype: object

Checking the % of missing values

In [19]:
# Count missing values
missing_counts = df.isnull().sum()

# Calculate percentage of missing data
missing_percentage = (missing_counts / len(df)) * 100
missing_percentage

gdp_per_capita    23.552575
continent          6.176721
new_cases          4.488689
new_deaths         4.384133
total_cases        4.105627
total_deaths       4.105627
iso_code           0.000000
location           0.000000
date               0.000000
population         0.000000
dtype: float64

## Data Cleaning

1. Missing Values
2. Coverting data types

### 1. Missing Data Treatment Plan

| Field | % Missing | Type | Handling | Reason |
|-------|-----------|------|----------|--------|
| `gdp_per_capita` | 23.55% | **MAR** | Median by continent | >15% missing; country characteristics affect reporting [Statology] |
| `continent` | 6.18% | **MAR** | Deterministic mapping by location + rule-based assignment | Mapping errors; depends on location coverage [OWID] |
| `new_cases` | 4.49% | **MCAR** | **Drop rows** | <5%; preserves epidemiological integrity [WHO] |
| `new_deaths` | 4.38% | **MCAR** | **Drop rows** | Same as new_cases |
| `total_cases` | 4.11% | **MCAR** | **Drop rows** | Consistent with daily metrics |
| `total_deaths` | 4.11% | **MCAR** | **Drop rows** | Consistent with daily metrics |

**References**:  
[1] [Statology: Handling Missing Data Decision Tree](https://www.statology.org/handling-missing-data-better-decision-tree-approach/) 
Decision tree for missing data handling based on % missing and data type.

[2] [OWID COVID-19 Data README](https://github.com/owid/covid-19-data/blob/master/public/data/README.md)
Official documentation explaining data collection and missingness patterns.

[3] [WHO COVID-19 Data Portal](https://www.who.int/data)
Guidelines on case/death reporting standards and data corrections.


#### Before Data Cleaning

In [20]:
df.isnull().sum()

gdp_per_capita    101143
continent          26525
new_cases          19276
new_deaths         18827
total_cases        17631
total_deaths       17631
iso_code               0
location               0
date                   0
population             0
dtype: int64

#### Dropping rows of:
- new_cases
- new_deaths
- total_cases
- total_deaths

In [21]:
cols = ['new_cases', 'new_deaths', 'total_cases', 'total_deaths']

df_clean = df.dropna(subset=cols).copy() # Create a clean copy after dropping rows with missing values in specified columns
print(f"Shape after Step 1: {df_clean.shape}")
print(f"Rows dropped: {len(df) - len(df_clean)} ({((len(df) - len(df_clean))/len(df)*100):.2f}%)")

Shape after Step 1: (410147, 10)
Rows dropped: 19288 (4.49%)


#### Imputation by location

**Deterministic Imputation Using Unique Mapping**

In [22]:
# Create reliable continent mapping from complete records
continent_map = (df_clean.dropna(subset=['continent']) # We only want rows where continent is known
                .drop_duplicates('location')[['location', 'continent']] # To ensure we get unigue continents per location
                .set_index('location')['continent'] # Make 'location' the index so we can easily convert to a mapping
                .to_dict())

print(continent_map)
# Apply mapping to fill missing continents
df_clean['continent'] = df_clean['location'].map(continent_map).fillna(df_clean['continent'])

print(f"Continent missing after Step 2: {df_clean['continent'].isnull().sum()}")

{'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'American Samoa': 'Oceania', 'Andorra': 'Europe', 'Angola': 'Africa', 'Anguilla': 'North America', 'Antigua and Barbuda': 'North America', 'Argentina': 'South America', 'Armenia': 'Asia', 'Aruba': 'North America', 'Australia': 'Oceania', 'Austria': 'Europe', 'Azerbaijan': 'Asia', 'Bahamas': 'North America', 'Bahrain': 'Asia', 'Bangladesh': 'Asia', 'Barbados': 'North America', 'Belarus': 'Europe', 'Belgium': 'Europe', 'Belize': 'North America', 'Benin': 'Africa', 'Bermuda': 'North America', 'Bhutan': 'Asia', 'Bolivia': 'South America', 'Bonaire Sint Eustatius and Saba': 'North America', 'Bosnia and Herzegovina': 'Europe', 'Botswana': 'Africa', 'Brazil': 'South America', 'British Virgin Islands': 'North America', 'Brunei': 'Asia', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa', 'Burundi': 'Africa', 'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Cape Verde': 'Africa', 'Cayman Islands': 'North America'

Since the missing count is still high, I will be filling in the values using location

In [23]:
# Get locations STILL missing continent + counts
missing_continent_summary = (
    df_clean[df_clean['continent'].isnull()]
    .groupby('location')
    .size()
    .sort_values(ascending=False)
    .reset_index()
)

missing_continent_summary.columns = ['location', 'missing_count']

print("LOCATIONS WITH MISSING CONTINENT:")
print(missing_continent_summary)
print(f"\nTotal missing continent records: {missing_continent_summary['missing_count'].sum()}")
print(f"Unique problematic locations: {len(missing_continent_summary)}")

LOCATIONS WITH MISSING CONTINENT:
                         location  missing_count
0                          Africa           1674
1                            Asia           1674
2                          Europe           1674
3             European Union (27)           1674
4           High-income countries           1674
5            Low-income countries           1674
6   Lower-middle-income countries           1674
7                   North America           1674
8                         Oceania           1674
9                   South America           1674
10  Upper-middle-income countries           1674
11                          World           1674

Total missing continent records: 20088
Unique problematic locations: 12


Based on above results

**Hierarchical / Aggregate-Level Mapping**

In [24]:
aggregate_mapping = {
    'Africa': 'Africa',
    'Asia': 'Asia', 
    'Europe': 'Europe',
    'European Union (27)': 'Europe',
    'High-income countries': 'High-income',
    'Low-income countries': 'Low-income',
    'Lower-middle-income countries': 'Lower-middle-income',
    'North America': 'North America',
    'Oceania': 'Oceania',
    'South America': 'South America',
    'Upper-middle-income countries': 'Upper-middle-income',
    'World': 'World'
}

# Apply aggregate mapping
df_clean['continent'] = df_clean['continent'].fillna(
    df_clean['location'].map(aggregate_mapping)
)

print(f"Continent missing after aggregate mapping: {df_clean['continent'].isnull().sum()}")

print("Unique continents now:")
sorted(df_clean['continent'].unique())

Continent missing after aggregate mapping: 0
Unique continents now:


['Africa',
 'Asia',
 'Europe',
 'High-income',
 'Low-income',
 'Lower-middle-income',
 'North America',
 'Oceania',
 'South America',
 'Upper-middle-income',
 'World']

### Imputation by continent (for gdp_per_capita)

In [25]:
print("Missing gdp_per_capita before:", df_clean['gdp_per_capita'].isna().sum())

df_clean['gdp_per_capita'] = df_clean.groupby('continent')['gdp_per_capita'].transform(
    lambda x: x.fillna(x.median() if x.notna().any() else np.nan)
)


print("Missing gdp_per_capita after:", df_clean['gdp_per_capita'].isna().sum())

Missing gdp_per_capita before:

 87043
Missing gdp_per_capita after: 6696


### After Data Cleaning

In [26]:
df_clean.isnull().sum()

gdp_per_capita    6696
continent            0
new_cases            0
new_deaths           0
total_cases          0
total_deaths         0
iso_code             0
location             0
date                 0
population           0
dtype: int64

### Drop the remaining incomplete rows

In [27]:
initial_missing = df_clean.isnull().sum().sum() # Get total missing values before final drop for all columns
df_clean = df_clean.dropna()
final_missing = df_clean.isnull().sum().sum() # Final missing values after drop

In [28]:
print(f"Shape after all cleaning: {df_clean.shape}")
print(f"Total rows dropped: {len(df) - len(df_clean)} ({((len(df) - len(df_clean))/len(df)*100):.2f}%)")
print(f"Remaining missing values: {final_missing}")

Shape after all cleaning: (403451, 10)
Total rows dropped: 25984 (6.05%)
Remaining missing values: 0


### 2. Data Type Conversion Plan (With Memory Optimization Reference)

| Field | Current Type | **Convert Type** | Decision | Reason |
|----------------|--------------|---------------------|----------------|----------------------|
| `continent` | object | **category** | **Yes** | Fixed levels (Africa, Asia, etc.). `category` saves ~80% memory vs `object` [1]. |
| `iso_code` | object | **category** | **Yes** | ~200 unique country codes. `category` optimizes memory (string → int mapping internally) [1]. |
| `location` | object | **category** | **Yes** | ~200 unique countries. Major memory savings for repeated grouping/filtering [1] |
| `date` | object | **datetime64[ns]** | **CRITICAL** | **MUST convert**. Enables time-series operations (resample, rolling, date filtering). |


**Reference**:

[1] [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/categorical.html#memory-usage)


#### Check memory usage of the dataframe before changing the dtype

In [29]:
# Check memory usage of the dataframe before changing dtypes
print("Memory usage before dtype changes:")
print(df_clean.info(memory_usage='deep'))

Memory usage before dtype changes:
<class 'pandas.core.frame.DataFrame'>
Index: 403451 entries, 0 to 429434
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   gdp_per_capita  403451 non-null  float64
 1   continent       403451 non-null  object 
 2   new_cases       403451 non-null  float64
 3   new_deaths      403451 non-null  float64
 4   total_cases     403451 non-null  float64
 5   total_deaths    403451 non-null  float64
 6   iso_code        403451 non-null  object 
 7   location        403451 non-null  object 
 8   date            403451 non-null  object 
 9   population      403451 non-null  int64  
dtypes: float64(5), int64(1), object(4)
memory usage: 108.4 MB
None


In [30]:
before_memory = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory usage AFTER (MB): {before_memory}")


Memory usage AFTER (MB): 108.43093395233154


#### 1. Convert categorical columns

In [31]:
# Convert categorical columns
cat_cols = ['continent', 'iso_code', 'location']
df_clean[cat_cols] = df_clean[cat_cols].astype('category')

print("\n✅ Categorical conversion complete")


✅ Categorical conversion complete


#### 2. Convert date to datetime 

In [32]:
# Convert to datetime and normalize to remove time component as time is not needed
df_clean['date'] = pd.to_datetime(df_clean['date'])

print("\n✅ Date conversion complete")


✅ Date conversion complete


#### Verify dtype

In [33]:
# VERIFY
print("\nData types AFTER conversion:")
print(df_clean.dtypes)


Data types AFTER conversion:
gdp_per_capita           float64
continent               category
new_cases                float64
new_deaths               float64
total_cases              float64
total_deaths             float64
iso_code                category
location                category
date              datetime64[ns]
population                 int64
dtype: object


In [34]:
print("\nUnique categories check:")
for col in cat_cols:
    print(f"{col}: {df_clean[col].nunique()} unique / {len(df_clean[col].cat.categories)} categories")

print("\nDate range check:")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")

print("\n✅ DATA TYPE CONVERSION COMPLETE - READY FOR ANALYSIS!")


Unique categories check:
continent: 7 unique / 7 categories
iso_code: 242 unique / 242 categories
location: 242 unique / 242 categories

Date range check:
Date range: 2020-01-05 00:00:00 to 2024-08-04 00:00:00

✅ DATA TYPE CONVERSION COMPLETE - READY FOR ANALYSIS!


#### Check memory usage of the dataframe after changing the dtype

In [35]:
# Check memory usage of the dataframe before changing dtypes
print("Memory usage before dtype changes:")
print(df_clean.info(memory_usage='deep'))

Memory usage before dtype changes:
<class 'pandas.core.frame.DataFrame'>
Index: 403451 entries, 0 to 429434
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   gdp_per_capita  403451 non-null  float64       
 1   continent       403451 non-null  category      
 2   new_cases       403451 non-null  float64       
 3   new_deaths      403451 non-null  float64       
 4   total_cases     403451 non-null  float64       
 5   total_deaths    403451 non-null  float64       
 6   iso_code        403451 non-null  category      
 7   location        403451 non-null  category      
 8   date            403451 non-null  datetime64[ns]
 9   population      403451 non-null  int64         
dtypes: category(3), datetime64[ns](1), float64(5), int64(1)
memory usage: 26.6 MB
None


In [36]:
after_memory = df_clean.memory_usage(deep=True).sum() / 1024**2

print(f"\nMemory usage AFTER (MB): {after_memory}")
print(f"Memory savings: {((before_memory - after_memory) / before_memory):.1%} reduction")


Memory usage AFTER (MB): 26.590479850769043
Memory savings: 75.5% reduction
